# 节点失败重试次数

In [5]:
from requests import HTTPError

from typing import TypedDict
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import RetryPolicy
from loguru import logger


# 1、定义状态
class EmptyState(TypedDict):
    pass


# 2、声明节点
def node_a(state: EmptyState) -> EmptyState:
    logger.info("node a正在运行")
    # 抛出异常
    raise HTTPError


# 3、构建图
builder = StateGraph(state_schema=EmptyState)
builder.add_node(
    "node_a",
    node_a,
    retry_policy=RetryPolicy(
        # 最大重试次数
        max_attempts=3,
        # 每次重试，间隔时间是否抖动
        jitter=False
    ))

builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)
graph = builder.compile()

try:
    graph.invoke({})
except HTTPError as e:
    logger.info("重试次数耗尽:{}", e)

2026-07-22 17:06:41.263 | INFO     | __main__:node_a:17 - node a正在运行
